# 1.1 张量与自动求导 / Tensors and automatic differentiation

手写一个 100 行的标量 autograd，弄清 loss.backward() 到底做了什么。

Write a 100-line scalar autograd and see what loss.backward() really does.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 手写迷你 autograd / A mini autograd by hand

补全 `Value` 类里的 6 个 TODO。前 5 个是各运算的局部导数，第 6 个是 `backward()` 的拓扑排序与反向遍历。测试会用同样的表达式分别在你的 `Value` 和 PyTorch 上计算，并逐个比较梯度。

Fill in the 6 TODOs in the `Value` class. The first five are the local derivatives of each operation, the sixth is the topological sort and reverse traversal in `backward()`. The tests evaluate the same expressions with your `Value` and with PyTorch and compare every gradient.

In [ ]:
import math


class Value:
    """A scalar that remembers how it was computed, so it can backpropagate."""

    def __init__(self, data, _children=(), _op=""):
        self.data = float(data)
        self.grad = 0.0                    # dL/d(self), filled in by backward()
        self._backward = lambda: None      # pushes self.grad to the inputs
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            # TODO 1: out = self + other. Push out.grad to self.grad and other.grad.
            raise NotImplementedError("TODO 1")

        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            # TODO 2: out = self * other.
            raise NotImplementedError("TODO 2")

        out._backward = _backward
        return out

    def __pow__(self, k):
        assert isinstance(k, (int, float)), "only constant exponents"
        out = Value(self.data ** k, (self,), f"**{k}")

        def _backward():
            # TODO 3: out = self ** k.
            raise NotImplementedError("TODO 3")

        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")

        def _backward():
            # TODO 4: out = tanh(self). Express the local derivative with t.
            raise NotImplementedError("TODO 4")

        out._backward = _backward
        return out

    def exp(self):
        e = math.exp(self.data)
        out = Value(e, (self,), "exp")

        def _backward():
            # TODO 5: out = exp(self).
            raise NotImplementedError("TODO 5")

        out._backward = _backward
        return out

    def backward(self):
        # TODO 6: (a) build a topological order of all nodes that self depends on,
        #         (b) set self.grad = 1.0,
        #         (c) call node._backward() from the output back to the leaves.
        raise NotImplementedError("TODO 6")

    # Everything below is built from the operations above. Nothing to change.
    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __truediv__(self, other): return self * (other ** -1 if isinstance(other, Value) else 1.0 / other)
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __rsub__(self, other): return Value(other) - self
    def __rtruediv__(self, other): return Value(other) / self

In [ ]:
# 检查 / checks
import torch

def _check(build, inputs, name):
    """Build the same expression with Value and with torch, then compare."""
    vs = [Value(x) for x in inputs]
    out = build(*vs)
    out.backward()
    ts = [torch.tensor(float(x), dtype=torch.float64, requires_grad=True) for x in inputs]
    ref = build(*ts)
    ref.backward()
    assert abs(out.data - ref.item()) < 1e-9, f"{name}: forward {out.data} vs torch {ref.item()}"
    for i, (v, t) in enumerate(zip(vs, ts)):
        assert abs(v.grad - t.grad.item()) < 1e-9, f"{name}: grad of input {i} is {v.grad}, torch says {t.grad.item()}"

# 1. the worked example from the text: L = (a*b + c) * a
_check(lambda a, b, c: (a * b + c) * a, [2.0, -3.0, 10.0], "worked example")
a, b, c = Value(2.0), Value(-3.0), Value(10.0)
L = (a * b + c) * a
L.backward()
assert (a.grad, b.grad, c.grad) == (-2.0, 4.0, 2.0), "worked example should give (-2, 4, 2)"

# 2. a node used several times: gradients must accumulate
_check(lambda a: a + a, [3.0], "a + a")
_check(lambda a, b: (a + b) * (a + b) * a, [1.5, -0.5], "reused node")

# 3. one tanh neuron
_check(lambda x1, x2, w1, w2, b: (x1 * w1 + x2 * w2 + b).tanh(), [2.0, 0.0, -3.0, 1.0, 6.88], "tanh neuron")

# 4. powers, division, subtraction, exp
_check(lambda a, b: (a ** 3 - b) / (a * b + 2.0), [1.2, 0.7], "pow and div")
_check(lambda a, b: (a * b).exp() + (2.0 * a - b).exp(), [0.3, -0.4], "exp")

# 5. a deeper chain, to make sure the order of the backward pass is right
def _deep(a, b):
    x = a
    for _ in range(5):
        x = (x * b + a).tanh() * x + b
    return x
_check(_deep, [0.4, -0.9], "deep chain")
print("m11_value: all tests passed")

<details><summary><b>提示 1 / Hint 1</b></summary>

每个 `_backward` 都是同一个句式，即 `输入.grad += 局部导数 * out.grad`。局部导数查推导里的那张表。

Every `_backward` has the same shape, `input.grad += local_derivative * out.grad`. Look up the local derivative in the table from the derivation.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

一定用 `+=`。测试里的 `a + a` 和 `(a+b)*(a+b)*a` 专门检查一个节点被多次使用的情况。

Use `+=`. The tests `a + a` and `(a+b)*(a+b)*a` check nodes that are used more than once.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

`backward()` 里用深度优先遍历，一个节点在它的所有输入都入列之后再入列，得到的列表倒过来就是需要的顺序。

In `backward()` use a depth-first traversal that appends a node after all of its inputs. The reversed list is the order you need.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import math


class Value:
    """A scalar that remembers how it was computed, so it can backpropagate."""

    def __init__(self, data, _children=(), _op=""):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            # Local derivative is 1 for both inputs. Use += because a node can
            # feed several consumers and their contributions add up.
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def __pow__(self, k):
        assert isinstance(k, (int, float)), "only constant exponents"
        out = Value(self.data ** k, (self,), f"**{k}")

        def _backward():
            self.grad += k * self.data ** (k - 1) * out.grad

        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")

        def _backward():
            self.grad += (1 - t * t) * out.grad

        out._backward = _backward
        return out

    def exp(self):
        e = math.exp(self.data)
        out = Value(e, (self,), "exp")

        def _backward():
            self.grad += e * out.grad

        out._backward = _backward
        return out

    def backward(self):
        topo, seen = [], set()

        def build(v):
            if v not in seen:
                seen.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)          # appended only after all of its inputs

        build(self)
        self.grad = 1.0                 # dL/dL
        for v in reversed(topo):        # output first, leaves last
            v._backward()

    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __truediv__(self, other): return self * (other ** -1 if isinstance(other, Value) else 1.0 / other)
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __rsub__(self, other): return Value(other) - self
    def __rtruediv__(self, other): return Value(other) / self
```

</details>

## 2. 用自己的 autograd 训练一个 MLP / Train an MLP with your own autograd

用上一题的 `Value` 实现神经元的前向计算和训练循环，让一个 2-4-1 的 tanh 网络学会 XOR。测试要求 300 步后均方误差低于 0.05，且四个输入的预测符号都正确。

Using `Value` from the previous exercise, implement the forward pass of a neuron and the training loop so that a 2-4-1 tanh network learns XOR. The tests require a mean squared error below 0.05 after 300 steps and the correct sign for all four inputs.

In [ ]:
import random

# This exercise uses the Value class from the previous exercise.


class Neuron:
    def __init__(self, n_in):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(n_in)]
        self.b = Value(0.0)

    def __call__(self, x):
        # TODO 1: return tanh(w . x + b) as a Value. x is a list of floats or Values.
        raise NotImplementedError("TODO 1")

    def parameters(self):
        return self.w + [self.b]


class Layer:
    def __init__(self, n_in, n_out):
        self.neurons = [Neuron(n_in) for _ in range(n_out)]

    def __call__(self, x):
        return [n(x) for n in self.neurons]

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]


class MLP:
    def __init__(self, n_in, sizes):
        dims = [n_in] + sizes
        self.layers = [Layer(dims[i], dims[i + 1]) for i in range(len(sizes))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x[0] if len(x) == 1 else x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


def train(model, xs, ys, steps=300, lr=0.1):
    """Full-batch gradient descent on the mean squared error.

    xs: list of input lists, ys: list of float targets.
    Returns the list of loss values (plain floats), one per step.
    """
    losses = []
    for _ in range(steps):
        # TODO 2: forward pass and mean squared error (a Value)
        # TODO 3: reset every p.grad to 0, then call loss.backward()
        # TODO 4: gradient step  p.data -= lr * p.grad
        # TODO 5: losses.append(loss.data)
        raise NotImplementedError("TODO 2-5")
    return losses

In [ ]:
# 检查 / checks
import random

random.seed(0)
xs = [[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]]
ys = [-1.0, 1.0, 1.0, -1.0]                     # XOR with targets in {-1, +1}

model = MLP(2, [4, 1])
assert len(model.parameters()) == 4 * 3 + 5, "a 2-4-1 MLP has 17 parameters"

n = Neuron(2)
n.w[0].data, n.w[1].data, n.b.data = 0.5, -1.0, 0.25
out = n([2.0, 1.0])
import math
assert abs(out.data - math.tanh(0.25)) < 1e-12, "Neuron should return tanh(w . x + b)"

losses = train(model, xs, ys, steps=300, lr=0.1)
assert len(losses) == 300 and all(isinstance(l, float) for l in losses), "return one float per step"
assert losses[-1] < losses[0], "the loss should go down"
assert losses[-1] < 0.05, f"final loss {losses[-1]:.4f} is too high. Did you reset the gradients every step?"
for x, y in zip(xs, ys):
    assert model(x).data * y > 0, f"wrong sign for input {x}"
print(f"m11_mlp: all tests passed (loss {losses[0]:.3f} -> {losses[-1]:.4f})")

<details><summary><b>提示 1 / Hint 1</b></summary>

神经元的输出是 `(w1*x1 + w2*x2 + b).tanh()`。从 `self.b` 开始逐项相加即可，`Value` 已支持与浮点数相乘。

A neuron outputs `(w1*x1 + w2*x2 + b).tanh()`. Start from `self.b` and add term by term. `Value` already supports multiplication by floats.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

训练循环每一步有四件事，顺序为前向求损失、把所有 `p.grad` 置 0、`loss.backward()`、`p.data -= lr * p.grad`。

Each step of the loop does four things in this order, namely forward and loss, set every `p.grad` to 0, `loss.backward()`, `p.data -= lr * p.grad`.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

如果 loss 先降后升或者不收敛，优先检查是否忘了清零梯度。

If the loss goes down and then up, or fails to converge, first check whether the gradients are being reset.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import random


class Neuron:
    def __init__(self, n_in):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(n_in)]
        self.b = Value(0.0)

    def __call__(self, x):
        act = self.b
        for wi, xi in zip(self.w, x):
            act = act + wi * xi
        return act.tanh()

    def parameters(self):
        return self.w + [self.b]


class Layer:
    def __init__(self, n_in, n_out):
        self.neurons = [Neuron(n_in) for _ in range(n_out)]

    def __call__(self, x):
        return [n(x) for n in self.neurons]

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]


class MLP:
    def __init__(self, n_in, sizes):
        dims = [n_in] + sizes
        self.layers = [Layer(dims[i], dims[i + 1]) for i in range(len(sizes))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x[0] if len(x) == 1 else x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


def train(model, xs, ys, steps=300, lr=0.1):
    losses = []
    for _ in range(steps):
        preds = [model(x) for x in xs]
        loss = sum((p - y) ** 2 for p, y in zip(preds, ys)) / len(ys)
        for p in model.parameters():
            p.grad = 0.0                # grads accumulate, so clear them first
        loss.backward()
        for p in model.parameters():
            p.data -= lr * p.grad
        losses.append(loss.data)
    return losses
```

</details>